Linear Regression (OLS) baseline
for GA-ANN manuscript revision.

Method:
- Raw 70/30 development/test split
- log1p transformation of suction
- StandardScaler fitted ONLY on development data
- OLS trained on development set
- Independent test evaluation
- CP > 50% subgroup evaluation

In [1]:
import os
import json
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ============================================================
# COLAB CONFIGURATION
# ============================================================

DATA_PATH = "/content/drive/MyDrive/PINNs/Suction_vsCP-modified_1.xlsx"

OUT_DIR = "/content/drive/MyDrive/NNsGA/revised_lr_results"

SEED = 42

os.makedirs(OUT_DIR, exist_ok=True)

In [4]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)

# ============================================================
# COLUMNS
# ============================================================

TARGET_COL = "Collapse Potential (%)"

FEATURE_COLS = [
    "Suction (kPa)",
    "Silica fume (%)",
    "Lime (%)",
    "Gypsum content (%)",
    "Applied vertical stress (kPa)",
    "Degree of Saturation (%)",
]

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_excel(DATA_PATH)

df = df[FEATURE_COLS + [TARGET_COL]].dropna().copy()

X = df[FEATURE_COLS].values.astype(np.float64)
y = df[TARGET_COL].values.astype(np.float64)

print("=" * 70)
print("LINEAR REGRESSION (OLS) BASELINE")
print("=" * 70)

print(f"Total observations: {len(df)}")

LINEAR REGRESSION (OLS) BASELINE
Total observations: 600


In [5]:
# ============================================================
# RAW 70/30 SPLIT
# ============================================================

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED
)

print(f"Development set: {len(X_dev)}")
print(f"Independent test set: {len(X_test)}")

Development set: 420
Independent test set: 180


In [6]:
# LEAKAGE-FREE PREPROCESSING

def preprocess_fit(X_train):
    X_train = X_train.copy()

    # Log transform suction because of its strong right skew
    X_train[:, 0] = np.log1p(np.clip(X_train[:, 0], 0, None))

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)

    return scaler, X_train_scaled


def preprocess_transform(X_data, scaler):
    X_data = X_data.copy()

    X_data[:, 0] = np.log1p(np.clip(X_data[:, 0], 0, None))

    return scaler.transform(X_data)


scaler, X_dev_scaled = preprocess_fit(X_dev)

X_test_scaled = preprocess_transform(
    X_test,
    scaler
)

In [7]:
# TRAIN OLS
model = LinearRegression()

model.fit(X_dev_scaled, y_dev)

LinearRegression()

In [8]:
# TEST PREDICTIONS
y_pred = model.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\nIndependent Test Results")
print("-" * 30)
print(f"RMSE : {rmse:.6f}")
print(f"MAE  : {mae:.6f}")
print(f"R²   : {r2:.6f}")


Independent Test Results
--------------------------------------------------
RMSE : 9.642442
MAE  : 4.801710
R²   : 0.515327


In [9]:
# CP > 50% ANALYSIS

high_mask = y_test > 50

high_results = {
    "n": int(np.sum(high_mask)),
    "RMSE": None,
    "MAE": None,
    "R2": None
}

if np.sum(high_mask) >= 2:

    y_high = y_test[high_mask]
    pred_high = y_pred[high_mask]

    high_results["RMSE"] = float(
        np.sqrt(mean_squared_error(y_high, pred_high))
    )

    high_results["MAE"] = float(
        mean_absolute_error(y_high, pred_high)
    )

    high_results["R2"] = float(
        r2_score(y_high, pred_high)
    )

print("\nCP > 50% Results")
print("-" * 50)
print(f"N    : {high_results['n']}")

if high_results["RMSE"] is not None:
    print(f"RMSE : {high_results['RMSE']:.6f}")
    print(f"MAE  : {high_results['MAE']:.6f}")
    print(f"R²   : {high_results['R2']:.6f}")
else:
    print("Insufficient observations for subgroup R².")


CP > 50% Results
--------------------------------------------------
N    : 6
RMSE : 35.265667
MAE  : 34.872993
R²   : -20.608118


In [10]:
# SAVE TEST PREDICTIONS

predictions_df = pd.DataFrame({
    "Observed_CP": y_test,
    "Predicted_CP": y_pred,
    "Residual": y_pred - y_test
})

predictions_df.to_csv(os.path.join(OUT_DIR, "linear_regression_test_predictions.csv"),index=False)

In [11]:
# SAVE SUMMARY
summary = {
    "model": "Ordinary Least Squares Linear Regression",
    "seed": SEED,
    "total_observations": int(len(df)),
    "development_n": int(len(X_dev)),
    "test_n": int(len(X_test)),
    "preprocessing": {
        "suction_transform": "log1p",
        "scaler": "StandardScaler",
        "scaler_fit_on": "development_set_only"
    },
    "test_metrics": {
        "RMSE": float(rmse),
        "MAE": float(mae),
        "R2": float(r2)
    },
    "CP_greater_than_50": high_results
}

with open(
    os.path.join(OUT_DIR, "linear_regression_results.json"),
    "w"
) as f:
    json.dump(summary, f, indent=2)

In [12]:
# SAVE COEFFICIENTS

coefficients_df = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "Coefficient": model.coef_
})

coefficients_df.to_csv(
    os.path.join(OUT_DIR, "linear_regression_coefficients.csv"),
    index=False
)

print("\nSaved files:")
print(os.path.join(OUT_DIR, "linear_regression_test_predictions.csv"))
print(os.path.join(OUT_DIR, "linear_regression_results.json"))
print(os.path.join(OUT_DIR, "linear_regression_coefficients.csv"))

print("\nDone.")


Saved files:
/content/drive/MyDrive/NNsGA/revised_lr_results/linear_regression_test_predictions.csv
/content/drive/MyDrive/NNsGA/revised_lr_results/linear_regression_results.json
/content/drive/MyDrive/NNsGA/revised_lr_results/linear_regression_coefficients.csv

Done.
